# Day 8 — JSON in Depth + Robust API Parsing

> ⚠️ **Why this matters.** Yesterday you got JSON back from an API. Today you learn to **survive bad JSON, missing fields, and weird shapes** — because real-world APIs return all of these. The difference between code that works on the happy path and code that ships is exactly this skill.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/08-json-parsing.ipynb)

## What you'll do today

**Time:** 90 min lesson + 60 min mini-project + 45 min quiz.

By the end:

- [ ] You can navigate any JSON structure safely
- [ ] You handle missing/None/optional fields without crashes
- [ ] You can use `try/except` and `.get()` chains effectively
- [ ] You've upgraded `api.py` to be paranoid-but-still-readable
- [ ] You know when to validate with Pydantic (preview)

## 1. JSON ↔ Python — the conversion table

| JSON | Python |
|------|--------|
| `{...}` | `dict` |
| `[...]` | `list` |
| `"hello"` | `str` |
| `42` | `int` |
| `3.14` | `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

That's everything. JSON has no dates, tuples, sets, or anything else — just these primitives. When you get a date from an API, it'll be a string you have to parse.

## 2. Safe traversal patterns

In [ ]:
# Imagine this came from an API:
response = {
    'word': 'thorough',
    'phonetics': [
        {'text': '/ˈθʌrə/', 'audio': 'https://example.com/audio.mp3'},
        {'text': '/ˈθʌroʊ/', 'audio': ''}
    ],
    'meanings': [
        {'partOfSpeech': 'adjective', 'definitions': [{'definition': 'complete'}]}
    ]
}

# Risky:
ipa = response['phonetics'][0]['text']    # KeyError or IndexError waiting
print(ipa)

In [ ]:
# Safe:
phonetics = response.get('phonetics', [])
ipa = phonetics[0]['text'] if phonetics else None
print(ipa)

**Rules:**

1. Every nested access is a chance to crash. Each `.get()` with a default protects one step.
2. For lists, check length OR use slicing (`x[0] if x else None`).
3. If the API contract is fixed, write a validator; if exploring, defend defensively.

## 3. The walrus operator (`:=`) for clarity

In [ ]:
# Walrus assigns inside an expression. Saves you a line and a name.
if (phonetics := response.get('phonetics')) and phonetics[0].get('text'):
    ipa = phonetics[0]['text']
    print('IPA:', ipa)

Use sparingly — it reads weird at first. Where it shines: `if (m := re.match(...)) is not None: ...`.

## 4. Loading and saving JSON files

In [ ]:
import json
from pathlib import Path

# Save
data = {'cached': {'thorough': {'ipa': '/ˈθʌrə/'}}}
Path('/tmp/cache.json').write_text(json.dumps(data, ensure_ascii=False, indent=2))

# Load
loaded = json.loads(Path('/tmp/cache.json').read_text())
print(loaded)

**Common mistakes:**

- Forgetting `ensure_ascii=False` (Thai → escaped)
- Forgetting `indent=` (one giant unreadable line)
- Catching `JSONDecodeError` and silently returning empty — better to crash loudly OR log + return default

**Pattern for cached file:**

In [ ]:
import json
from pathlib import Path

def load_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError as e:
        print(f'Corrupt cache at {path}: {e}')
        return {}

def save_cache(path: Path, data: dict) -> None:
    path.parent.mkdir(exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2))

## 5. Preview — Pydantic (Phase 3 will cover this)

For Phase 3, you'll use Pydantic to validate API responses with one line per field:

```python
from pydantic import BaseModel

class Phonetic(BaseModel):
    text: str
    audio: str = ''

class Entry(BaseModel):
    word: str
    phonetics: list[Phonetic] = []

entry = Entry.model_validate(api_response)
# Now everything is typed, validated, and IDE-completes
```

We're not using Pydantic in english-helper yet — keeping dependencies minimal.

## End-of-day mini-project — robust `api.py`

> 🎯 **Today's piece:** harden yesterday's API module against missing fields, weird shapes, and bad JSON.

### What you're building

Refactor `src/english_helper/api.py` from Day 7 so:

- Every function works even when fields are missing, empty, or unexpected types.
- `fetch_word()` returns a normalized dict with exactly these keys: `word`, `ipa`, `definition`, `audio`. Each may be empty string or None.
- Add a function `to_word_entry(api_data)` that converts API response → english-helper's internal `WordEntry` format.
- Add a test (just an `assert` in `__main__`) that calls each function with empty `{}`, `{'phonetics': []}`, and `{'phonetics': [{}]}` and shows none of them crashes.

### Try it

In [ ]:
# Sketch here. Then write the real code in api.py.

def to_word_entry(api_data: dict) -> dict[str, str]:
    # Your code — return {'word': ..., 'ipa': ..., 'definition': ...}
    ...


<details>
<summary>Solution</summary>

```python
# Excerpt — full file follows the same pattern
def to_word_entry(api_data: dict) -> dict[str, str]:
    return {
        'word':       api_data.get('word', ''),
        'ipa':        extract_ipa(api_data) or '',
        'definition': extract_definition(api_data) or '',
        'audio':      extract_audio_url(api_data) or '',
    }

if __name__ == '__main__':
    # Defensive tests — should print 4 dicts with empty strings, no crashes
    print(to_word_entry({}))
    print(to_word_entry({'word': 'x'}))
    print(to_word_entry({'phonetics': []}))
    print(to_word_entry({'phonetics': [{}]}))
```
</details>

## Connect to the project

> 🎯 **Connects to the project:** Tomorrow (Day 9) you build the cache that makes english-helper fast and offline-resilient. With today's defensive parsing, Day 9's cache can store the normalized form — much smaller and more usable than the raw API response.

**Quiz:** [08-json-parsing-quiz.ipynb](08-json-parsing-quiz.ipynb)